In [197]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import floor
import csv

In [198]:
def sigmoid(inpLst):
    return 1/(1+np.exp(-inpLst))
def ReLU(x_):
    return x_ if x_ > 0 else 0
def np_ReLU(x_):
    return np.maximum(0,x_)
def leaky_ReLU(x_, alpha=0.01):
    return np.where(x_ > 0, x_, x_ * alpha)
def softmax(x_):
    return np.exp(x_ - np.max(x_, keepdims=True)) / np.sum(np.exp(x_ - np.max(x_, keepdims=True)), keepdims=True)

In [199]:
class neuronCell:#神经元（单层）    Neuron Layer
    def __init__(self,inpDim_,optDim_):
        self.inputDim=inpDim_
        self.outputDim=optDim_
        self.weights = np.random.randn(inpDim_, optDim_) * np.sqrt(2.0 / inpDim_)
        self.b = np.zeros((1, optDim_))
        #参数   Parametres
        self.x = None   #输入           Input
        self.y = None   #矩阵乘法结果   Result of xW+b 
        self.z = None   #输出           Output
        self.dx = None  #梯度输入       Gradient of Input
        self.dw = None  #梯度权重       Gradient of Weight
        self.db = None  #梯度偏置       Gradient of b

    def vowart(self):
        raise NotImplementedError(f"类{self.__class__.__name__}没有前向传播函数")

    def backwart(self):
        raise NotImplementedError(f"类{self.__class__.__name__}没有后向传播函数")
    
    def meanSquareError(self, target_,average=True):#均方差/平方和误差
        diff = self.z - target_
        N = self.z.shape[0]
        if average:
            loss = np.sum(diff**2) / N
            dout = 2 * diff / N
        else:
            loss = np.sum(diff**2)
            dout = 2 * diff
        return loss, dout
    
    def clear(self):
        self.x = None
        self.z = None
        self.dx = None
        self.dw = None
        self.db = None

class linCell(neuronCell):#线性神经元
    def __init__(self,inpDim_,optDim_):
        super().__init__(inpDim_,optDim_)

    def vowart(self, inp_):
        inp_ = np.array(inp_)
        if inp_.ndim == 1:
            if inp_.size % self.inputDim != 0:
                raise ValueError(f"输入元素个数 {inp_.size} 无法整除输入维度 {self.inputDim}")
            inp_ = inp_.reshape(-1, self.inputDim)
        self.x = inp_
        self.z = np.dot(inp_, self.weights) +self.b
        return self.z

    def backwart(self, inp_):
        self.dw = np.dot(self.x.T, inp_)
        self.db = np.sum(inp_, axis=0, keepdims=True)
        self.dx = np.dot(inp_, self.weights.T)
        return self.dx
    
    def meanSquareError(self, target_,average=True):
        if target_.ndim == 1:
            target_ = target_.reshape(-1, 1)
        return super().meanSquareError(target_,average)
    
class ActivationCell:#激活层
    def __init__(self):
        self.x = None
        self.z = None
        self.dx = None

    def vowart(self, inp_):
        raise NotImplementedError

    def backwart(self, dout):
        raise NotImplementedError

    def clear(self):
        self.x = None
        self.z = None
        self.dx = None

class ReLUCell(ActivationCell):#ReLU激活层
    def vowart(self, inp_):
        self.x = inp_
        self.z = np_ReLU(inp_)
        return self.z

    def backwart(self, dout):
        self.dx = dout * (self.x > 0)
        return self.dx
    
class sigmoidCell(ActivationCell):#sigmoid激活层
    def vowart(self, inp_):
        self.x = inp_
        self.z = sigmoid(inp_)
        return self.z

    def backwart(self, dout):
        local_grad = self.z * (1 - self.z)
        self.dx = dout * local_grad
        return self.dx

class softmaxCell(ActivationCell):#softmax激活层
    def vowart(self, inp_):
        self.x = inp_
        self.z = softmax(inp_)
        return self.z

    def backwart(self, dout):
        p = self.z
        # 计算 sum_j dout_j * p_j，即逐样本的加权和
        sum_dout_p = np.sum(dout * p, axis=1, keepdims=True)
        self.dx = p * (dout - sum_dout_p)
        return self.dx

class leakyReLUCell(ActivationCell):#ReLU激活层
    def vowart(self, inp_):
        self.x = inp_
        self.z = leaky_ReLU(inp_)
        return self.z

    def backwart(self, dout,alpha=0.01):
        self.dx = np.where(self.x > 0, 1.0, alpha)*dout
        return self.dx

LAYER_REGISTRY = {
    "linear": linCell,  
    "relu": ReLUCell,
    "sigmoid": sigmoidCell,
    "softmax": softmaxCell,
    "leakyrelu": leakyReLUCell
}
class neuralNet:#神经网络   最后一层必须是线性层
    def __init__(self,typeLayers_,inpDims_,optDims_):
        self.layers=[]
        indexx=0
        for i in range(len(typeLayers_)):
            layerName = typeLayers_[i].lower()
            if layerName == "linear":
                self.layers.append(LAYER_REGISTRY[layerName](inpDims_[indexx],optDims_[indexx]))
                indexx+=1
            else:
                self.layers.append(LAYER_REGISTRY[layerName]())
            
    def vowart(self, inp_):
        inp_ = np.array(inp_)
        for layer in self.layers:
            inp_ = layer.vowart(np.array(inp_))
        return inp_

    def backwart(self, inp_):
        for layer in reversed(self.layers):
            inp_ = layer.backwart(np.array(inp_))
        return inp_

    def clear_cache(self):
        for layer in self.layers:
            layer.clear_cache()

    def train(self,inp_, target_,learningRate):#训练逻辑：前向传播，计算损失和输入梯度，反向传播，梯度下降
        predict = self.vowart(inp_)
        loss, dout = self.layers[-1].meanSquareError(target_, average=True)
        self.backwart(dout)
        for layer in self.layers:
            if hasattr(layer, 'weights') and hasattr(layer, 'dw'):
                layer.weights -= learningRate * layer.dw
                layer.b -= learningRate * layer.db
        return loss

    def meanSquareError(self,inp_, target_,average=True):#均方差/平方和误差
        tmp = self.vowart(inp_)
        diff = tmp - target_
        N = tmp.shape[0]
        if average:
            loss = np.sum(diff**2/ N) 
        else:
            loss = np.sum(diff**2)
        return loss
    
    def save(self, filename="neuralnet.npz"):
        save_dict = {}
        for idx, layer in enumerate(self.layers):
            if hasattr(layer, 'weights') and hasattr(layer, 'b'):
                save_dict[f"layer_{idx}_type"] = "linear"
                save_dict[f"layer_{idx}_input_dim"] = layer.inputDim
                save_dict[f"layer_{idx}_output_dim"] = layer.outputDim
                save_dict[f"layer_{idx}_weights"] = layer.weights
                save_dict[f"layer_{idx}_bias"] = layer.b
            else:
                layer_type = None
                for key, cls in LAYER_REGISTRY.items():
                    if isinstance(layer, cls):
                        layer_type = key
                        break
                if layer_type is None:
                    raise ValueError(f"未注册的层类型: {layer.__class__.__name__}")
                save_dict[f"layer_{idx}_type"] = layer_type
        np.savez(filename, **save_dict)
        
    def load(self, filename="neuralnet.npz"):
        data = np.load(filename)
        self.layers = []                # 直接清空实例的层列表
        idx = 0
        while f"layer_{idx}_type" in data:
            l_type = str(data[f"layer_{idx}_type"])
            if l_type == "linear":
                inp_dim = int(data[f"layer_{idx}_input_dim"])
                out_dim = int(data[f"layer_{idx}_output_dim"])
                layer = linCell(inp_dim, out_dim)
                layer.weights = data[f"layer_{idx}_weights"]
                layer.b = data[f"layer_{idx}_bias"]
                self.layers.append(layer)   # 直接添加到实例的 layers
            else:
                self.layers.append(LAYER_REGISTRY[l_type.lower()]())
            idx += 1

提取数据

In [200]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")
feature_cols = ["np","nn","delta","I", "betaMinusQ", "betaMinusQUncertainty", "halflife_log_sec_uncertainty"]
#feature_cols = ["np", "na", "betaMinusQ", "betaMinusQUncertainty"]
target_cols = ["halflife_log_sec"]

trainX = train_df[feature_cols].values
trainY = train_df[target_cols].values

validX = valid_df[feature_cols].values
validY = valid_df[target_cols].values

testX = test_df[feature_cols].values
testY = test_df[target_cols].values

输入归一化

In [201]:
#核子数类：np(第0列), na(第1列) 除以 118
trainX = trainX.copy().astype(float)
validX = validX.copy().astype(float)
testX = testX.copy().astype(float)

trainX[:, 0] /= 118.0   # np
trainX[:, 1] /= 118.0   # na
validX[:, 0] /= 118.0
validX[:, 1] /= 118.0
testX[:, 0] /= 118.0
testX[:, 1] /= 118.0

#其余特征（从第2列开始）除以三个集合的最大值
#拼接三个集合，计算每列最大值
other_cols_start = 2   # 假设第0列np，第1列na，其余从第2列起
all_other = np.vstack([trainX[:, other_cols_start:],
                       validX[:, other_cols_start:],
                       testX[:, other_cols_start:]])
max_vals = np.max(all_other, axis=0)

#对各集合的其余特征除以对应最大值
trainX[:, other_cols_start:] /= max_vals
validX[:, other_cols_start:] /= max_vals
testX[:, other_cols_start:] /= max_vals

建立神经网络

In [202]:
NNeurons = 32
NLayers = 6
dim_Input =len(feature_cols)
dim_Target = len(target_cols)
inp = [dim_Input]
opt = [NNeurons]
types = ["linear","leakyrelu"]
for i in range(NLayers-2):
    types.append("linear")
    types.append("leakyrelu")
    inp.append(NNeurons)
    opt.append(NNeurons)
types.append("linear")
inp.append(NNeurons)
opt.append(dim_Target)
neural=neuralNet(types,inp,opt)

训练

In [203]:
mode = int(input("输出模式，0为从零训练，1为继续训练，2为读取数据并拟合"))

epsilon = 0.03#定义容许误差 Определение допустимой погрешности
maxTime = int(1E5)
alpha = 1E-3
best_lo_validation = 1E20
patience = 100#如果验证集误差持续不降低则失去耐心停止训练
best_weights = []
best_biases = []
timeTrained=0

if mode:
    neural.load()

if mode == 1 or mode == 0:
    print(f"Среднеквадратичная ошибка до обучения: {neural.meanSquareError(trainX, trainY)}")
if mode ==1 or mode ==0:
    for i in range(maxTime):
        lo = neural.train(trainX, trainY, alpha)
        timeTrained+=1
        lo_validation = neural.meanSquareError(validX,validY)
        if best_lo_validation > lo_validation:
            best_lo_validation = lo_validation
            best_weights = []
            best_biases = []
            for layer in neural.layers:
                if hasattr(layer, 'weights'):
                    best_weights.append(layer.weights.copy())
                    best_biases.append(layer.b.copy())
                else:
                    best_weights.append(None)
                    best_biases.append(None)
            patience = 20
        else:
            patience -= 1
        if lo < epsilon or patience < 0:
            break
    neural.save()
if mode == 1 or mode == 0:
    print(f"Количество итераций обучения: {timeTrained}")
    print(f"Среднеквадратичная ошибка после обучения: {neural.meanSquareError(trainX, trainY)}")
else:
    print(f"Среднеквадратичная ошибка для обучающей выборки: {neural.meanSquareError(trainX, trainY)}")
print(f"Среднеквадратичная ошибка для тестовой выборки: {neural.meanSquareError(testX, testY)}")
print(f"Среднеквадратичная ошибка для валидационной выборки: {neural.meanSquareError(validX, validY)}")

Среднеквадратичная ошибка до обучения: 12.78004839581399
Количество итераций обучения: 684
Среднеквадратичная ошибка после обучения: 5.715064857746929
Среднеквадратичная ошибка для тестовой выборки: 1.3209742577203942
Среднеквадратичная ошибка для валидационной выборки: 5.417796587201904
